<a href="https://colab.research.google.com/github/Heba2627/NLP-Project/blob/main/NLP_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Importing Dataset via. Pandas

In [19]:
import pandas as pd
import numpy as np
import regex as re
import warnings
warnings.filterwarnings("ignore")

# Login using e.g. `huggingface-cli login` to access this dataset
splits = {'train': 'train.parquet', 'validation': 'validation.parquet', 'test': 'test.parquet'}
dfs=[]
for splitname,path in splits.items():
    df=pd.read_parquet("hf://datasets/coastalcph/tydi_xor_rc/"+ path)
    df["split"]=splitname
    dfs.append(df)

df=pd.concat(dfs, ignore_index=True)

In [20]:
df=df[df["lang"].isin(["ar", "ko", "te"])]

In [ ]:
print(df.columns)
answerable_df = df[df["answerable"] == True]
unanswerable_df = df[df["answerable"] == False]

print(f"Number of answerable instances: {len(answerable_df)}")
print(f"Number of unanswerable instances: {len(unanswerable_df)}")
print(f"{len(answerable_df)/len(df)*100:.2f}% of dataset that is answerable")
print(f"{len(unanswerable_df)/len(df)*100:.2f}% of dataset that is unanswerable")

Index(['question', 'context', 'lang', 'answerable', 'answer_start', 'answer',
       'answer_inlang', 'split'],
      dtype='object')
Number of answerable instances: 6963
Number of unanswerable instances: 527
92.96% of dataset that is answerable
7.04% of dataset that is unanswerable


In [ ]:
#Check for letters in all languages + Mark for arabic
train_df= df[df["split"] == "train"]
val_df= df[df["split"] == "validation"]

word_pattern = re.compile(r"[\p{L}\p{M}]+", flags=re.UNICODE)

def n_tokens(text):
    return len(word_pattern.findall(str(text)))
def median_iqr(s):
    q1, median, q3 = np.percentile(s, [25, 50, 75])
    return pd.Series({"median": median, "Q1": q1, "Q3": q3, "IQR": q3 - q1})

df["question_len"] = df["question"].apply(n_tokens)
df["context_len"] = df["context"].apply(n_tokens)

for split_df in [train_df, val_df]:
    split_df["question_len"] = split_df["question"].apply(n_tokens)
    split_df["context_len"] = split_df["context"].apply(n_tokens)


def corpus_stats_by_lang(split_df, split_name):
    rows = []
    for lang, g in split_df.groupby("lang"):
        q_stats = median_iqr(g["question_len"])
        c_stats = median_iqr(g["context_len"])
        rows.append({
            "split": split_name,
            "lang": lang,
            "n": len(g),
            "pct_answerable": g["answerable"].mean() * 100,
            "q_median_iqr": f'{q_stats["median"]:.1f} ({q_stats["IQR"]:.1f})',
            "c_median_iqr": f'{c_stats["median"]:.1f} ({c_stats["IQR"]:.1f})',
        })

    # "All languages" summary row for this split
    q_all = median_iqr(split_df["question_len"])
    c_all = median_iqr(split_df["context_len"])
    rows.append({
        "split": split_name,
        "lang": "All",
        "n": len(split_df),
        "pct_answerable": split_df["answerable"].mean() * 100,
        "q_median_iqr": f'{q_all["median"]:.1f} ({q_all["IQR"]:.1f})',
        "c_median_iqr": f'{c_all["median"]:.1f} ({c_all["IQR"]:.1f})',
    })
    return pd.DataFrame(rows)


combined_stats = pd.concat([
    corpus_stats_by_lang(train_df, "train"),
    corpus_stats_by_lang(val_df, "validation"),
], ignore_index=True)

print(combined_stats.to_string(index=False))


# ── Missing values and duplicates, reported per split (not per language) ──
for split_name, split_df in [("train", train_df), ("validation", val_df)]:
    print(f"\n{split_name}:")
    print(split_df.isna().sum())
    n_dup_rows = split_df.duplicated(subset=["question", "context"], keep="first").sum()
    print(f"redundant duplicate rows: {n_dup_rows}")

     split lang    n  pct_answerable q_median_iqr c_median_iqr
     train   ar 2558       90.031274    6.0 (2.0)  86.5 (67.0)
     train   ko 2422       97.398844    5.0 (2.0)  82.0 (63.0)
     train   te 1355       96.678967    5.0 (2.0)  77.0 (60.0)
     train  All 6335       94.269929    5.0 (3.0)  83.0 (64.0)
validation   ar  415       87.469880    6.0 (2.0)  84.0 (64.5)
validation   ko  356       94.662921    5.0 (2.0)  81.0 (71.2)
validation   te  384       75.781250    6.0 (2.0) 100.0 (63.2)
validation  All 1155       85.800866    5.0 (2.5)  89.0 (69.0)

train:
question            0
context             0
lang                0
answerable          0
answer_start        0
answer              0
answer_inlang    6285
split               0
question_len        0
context_len         0
dtype: int64
redundant duplicate rows: 10

validation:
question            0
context             0
lang                0
answerable          0
answer_start        0
answer              0
answer_inlang    1

In [ ]:
#missing values
missing = df.isna().sum().to_frame("n_missing")
missing["pct_missing"] = (100 * missing["n_missing"]/len(df))
missing

#duplicates
dup_filter = df.duplicated(subset=["question", "context"], keep=False)
dup_amount = dup_filter.sum()
dup_groups = df[dup_filter].groupby(["question","context"]).ngroups
redudant_rows = df.duplicated(subset=["question", "context"], keep="first").sum()
print(dup_groups)
print(dup_amount)
print(redudant_rows)

10
20
10


# Tokenisation

In [21]:
from transformers import AutoTokenizer,AutoModelForSeq2SeqLM

model_name = "facebook/nllb-200-distilled-600M"
tok = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to("cuda")

LANG_MAP = {"ar": "arb_Arab","ko": "kor_Hang","te": "tel_Telu"}
def translate(token, lang):
    try:
        tok.src_lang = LANG_MAP[lang]
        inputs = tok(token, return_tensors="pt").to("cuda")
        output = model.generate(**inputs,forced_bos_token_id=tok.convert_tokens_to_ids("eng_Latn"))
        return tok.decode(output[0], skip_special_tokens=True)

    except Exception as e:
        print(f"failed: {token!r} ({lang}) -> {type(e).__name__}: {e}")
        return None

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

In [22]:
translation_cache = {}

def translate_cached(text, lang):
    key = (text, lang)
    if key in translation_cache:
        return translation_cache[key]
    result = translate(text, lang)
    translation_cache[key] = result
    return result

In [ ]:
#NOTE: I will leave this for now and then when I just need to test I will go back here and translate
unique_questions = df[["question", "lang"]].drop_duplicates()
for q, lang in zip(unique_questions["question"], unique_questions["lang"]):
    translate_cached(q, lang)

In [23]:
from collections import Counter, defaultdict

#We need to root out symbols so check for letter
has_letter = re.compile(r"\p{L}", flags=re.UNICODE)

def tokenize_words_xlmr(text):
    pieces = tok.tokenize(str(text))
    words, current = [], ""
    for p in pieces:
        if p.startswith("▁"):
            if current != "":
                words.append(current)
            current = p[1:]
        else:
            current += p
    if current != "":
        words.append(current)
    return [w.lower() for w in words if w != "" and has_letter.search(w)]

counters = defaultdict(Counter)
for lang, question in zip(df["lang"], df["question"]):
    counters[lang].update(tokenize_words_xlmr(question))

rows = [
    {"lang": lang, "token": token, "count": count}
    for lang, counter in counters.items()
    for token, count in counter.most_common(5)
]
top5_df = pd.DataFrame(rows)
top5_df = top5_df[top5_df["lang"].isin(["ar", "ko", "te"])]
top5_df["In English"] = [translate(token, lang) for token, lang in zip(top5_df["token"], top5_df["lang"])]
top5_df = top5_df[["lang", "token", "In English", "count"]]
top5_df

,lang,token,In English,count
0,ko,가장,The most,593
1,ko,무엇인가?,What?,570
2,ko,언제,When?,380
3,ko,어디인가?,Where are you?,257
4,ko,몇,A few.,255
5,ar,من,Who ?,697
6,ar,في,In the,682
7,ar,متى,When ?,600
8,ar,ما,What ?,522
9,ar,هو,It is .,415


In [ ]:
#Task 3
def verify_answerable(df):
    checked = 0
    failures = []
    for i, row in enumerate(df):
        if row["answerable"]:
            checked+= 1
            context=row["context"]
            answer=row["answer"]
            start=row["answer_start"]

            expected=context[start:start+len(answer)]
            if expected!=answer:
                failures.append({
                    "index": i,
                    "answer_start": start,
                    "answer": answer,
                    "substring_found": expected,
                    "context": context})
    return checked,failures

#The pdf said to apply it to both splits
#we have to use ar,ko and te only, and they exist only in train and validation.
checked_train,failures_train=verify_answerable(train_df.to_dict("records"))
print(f"Checked {checked_train} answerable rows in the training set, found {len(failures_train)} failures")

checked_val,failures_val=verify_answerable(val_df.to_dict("records"))
print(f"Checked {checked_val} answerable rows in the validation set, found {len(failures_val)} failures")


Checked 5972 answerable rows in the training set, found 0 failures
Checked 991 answerable rows in the validation set, found 0 failures


# i) Majority Baseline

In [ ]:
majority_label = df["answerable"].value_counts().idxmax()
print("Majority class form training split: ", majority_label)

def majority_baseline(question=None, Context=None):
    return majority_label

Majority class form training split:  True


In [ ]:
val_pred = [majority_baseline(q, c) for q, c in zip(val_df["question"], val_df["context"])]
acc_score = (pd.Series(val_pred).values == val_df["answerable"].values).mean()
print("Accuracy of Majority Baseline on Validation split: ", acc_score*100, "%")

Accuracy of Majority Baseline on Validation split:  85.8008658008658 %


## ii) Determinstic rule-based classifier baseline

- Raw tokenization overlapping between questions and contexts would never work(different languages)
- Whatever approach I take will struggle with the semintically equivalent words like created and invented, as we can't have semantic understanding without a model(unless I bring a synonym dictionary)
- What I am going to do is to basically translate the question using deep-translator's google translator, tokenize the question and the context, removing stop words, put into lower case, and possibly lemmatize, and then I will depend on lexical overlap between the tokens but not the pure thing(because of the semantically equivalent words problem), so I will try to map some question words to their expected answers using regex, and then I will use the overlap score between the question and the context to determine the answerability.
- What it can exploit is the samples that survives the translation, named entities, numbers, dates, and question-word typing. These will survive because proper nouns and numerals are often unchanged, and question-word categories are a small closed set you can pattern-match directly.
- What it can't exploit is the mistranslations, negations and basically anyhting requires semantic understanding.

In [ ]:
import string

tokenizer= AutoTokenizer.from_pretrained("xlm-roberta-base")

STOPWORDS= {
    "the", "a", "an", "is", "are", "was", "were", "of", "in", "on", "at",
    "to", "for", "and", "or", "that", "this", "these", "those", "by", "with",
    "from", "as", "it", "its"
}

QUESTION_TYPE_PATTERNS= {
    "person":re.compile(r"\bwho\b", re.I),
    "date":re.compile(r"\bwhen\b", re.I),
    "location":re.compile(r"\bwhere\b", re.I),
    "number":re.compile(r"\bhow many\b|\bhow much\b", re.I),
}

ENTITY_PATTERNS= {
    "person":re.compile(r"\b[A-Z][a-z]+(?:\s[A-Z][a-z]+)+\b"),
    "date":re.compile(r"\b\d{1,2}(st|nd|rd|th)?\s\w+\s\d{4}\b|\b\d{4}\b|\b\w+\s\d{1,2},\s\d{4}\b"),
    "location":re.compile(r"\b[A-Z][a-z]+\b"),
    "number":re.compile(r"\b\d+([.,]\d+)?\b"),
}


def clean_tokens(text):
    tokens= tokenizer.tokenize(text.lower())
    tokens= [token for token in tokens if token and token not in string.punctuation]
    tokens= [token for token in tokens if token not in STOPWORDS]
    return set(tokens)


def lexical_overlap_score(question_en, context):
    q_tokens= clean_tokens(question_en)
    c_tokens= clean_tokens(context)
    if not q_tokens:
        return 0.0
    return len(q_tokens&c_tokens)/len(q_tokens)


def detect_question_type(question_en):
    for qtype, pattern in QUESTION_TYPE_PATTERNS.items():
        if pattern.search(question_en):
            return qtype
    return "other"


def context_has_type_match(context, qtype):
    if qtype == "other":
        return False
    pattern= ENTITY_PATTERNS.get(qtype)
    return bool(pattern.search(context)) if pattern else False


def predict_answerable(question, context, source_lang="auto"):
    question_en= translate(question, lang=source_lang)
    overlap= lexical_overlap_score(question_en, context)
    qtype= detect_question_type(question_en)
    type_match= context_has_type_match(context, qtype)
    prediction= (overlap>=0.1) or type_match
    return prediction

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
validation_df=val_df.copy()

def evaluate_classifier(val_df):
    y_true, y_pred= [], []

    for row in val_df.to_dict("records"):
        pred = predict_answerable(
            row["question"], row["context"],
            source_lang=row["lang"],
        )
        y_true.append(bool(row["answerable"]))
        y_pred.append(pred)



    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
    }
    return metrics



metrics= evaluate_classifier(validation_df)
print(f"Accuracy:{metrics['accuracy']:.3f}")
print(f"Precision:{metrics['precision']:.3f}")
print(f"Recall:{metrics['recall']:.3f}")
print(f"F1:{metrics['f1']:.3f}")
print(f"Confusion matrix [[TN, FP], [FN, TP]]: {metrics['confusion_matrix']}")

#NOTE: Check the max seq length problem

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (611 > 512). Running this sequence through the model will result in indexing errors


Accuracy:0.855
Precision:0.859
Recall:0.994
F1:0.921
Confusion matrix [[TN, FP], [FN, TP]]: [[2, 162], [6, 985]]


# Week 37:

## BIO Labeling of the training split

In [ ]:
#CODE CELL WAS IMPORTED FROM lab2.ipynb
TOKEN_PATTERN = re.compile(r"\w+|[^\w\s]", re.UNICODE)


def tokenize_with_offsets(text):
    matches = list(TOKEN_PATTERN.finditer(text))
    tokens = [match.group(0) for match in matches]
    offsets = [(match.start(), match.end()) for match in matches]
    return tokens, offsets


def character_span_to_bio(context, answer_start=None, answer_text=""):
    tokens, offsets = tokenize_with_offsets(context)
    labels = ["O"] * len(tokens)

    if answer_start is None or answer_text == "":
        return tokens, offsets, labels

    answer_end = answer_start + len(answer_text)
    if context[answer_start:answer_end] != answer_text:
        raise ValueError("The supplied answer text does not match the character span")

    covered = [
        index
        for index, (start, end) in enumerate(offsets)
        if start < answer_end and end > answer_start
    ]
    if not covered:
        raise ValueError("The answer does not overlap any token")
    if offsets[covered[0]][0] != answer_start or offsets[covered[-1]][1] != answer_end:
        raise ValueError("The answer span does not align with token boundaries")

    labels[covered[0]] = "B-ANS"
    for index in covered[1:]:
        labels[index] = "I-ANS"
    return tokens, offsets, labels


def bio_to_character_span(context, offsets, labels):
    if len(offsets) != len(labels):
        raise ValueError("Offsets and labels must have the same length")
    if not set(labels) <= {"O", "B-ANS", "I-ANS"}:
        raise ValueError("Only O, B-ANS and I-ANS labels are supported")

    answer_indices = [
        index for index, label in enumerate(labels) if label != "O"
    ]
    if not answer_indices:
        return None, ""

    start_index = answer_indices[0]
    expected_indices = list(range(start_index, start_index + len(answer_indices)))
    expected_labels = ["B-ANS"] + ["I-ANS"] * (len(answer_indices) - 1)
    if answer_indices != expected_indices:
        raise ValueError("The labels contain multiple or non-contiguous answer spans")
    if [labels[index] for index in answer_indices] != expected_labels:
        raise ValueError("The answer span must begin with B-ANS and continue with I-ANS")

    start = offsets[answer_indices[0]][0]
    end = offsets[answer_indices[-1]][1]
    return start, context[start:end]

In [ ]:
df = df.copy()
df = df.drop(columns=["tokens", "offsets", "labels", "alignable"], errors="ignore")

def try_bio(row):
    if not row["answerable"]:
        # Unanswerable: always succeeds, labels are all "O"
        tokens, offsets, labels = character_span_to_bio(row["context"], None, "")
        return pd.Series({"tokens": tokens, "offsets": offsets, "labels": labels, "alignable": True})
    try:
        tokens, offsets, labels = character_span_to_bio(
            row["context"], row["answer_start"], row["answer"]
        )
        return pd.Series({"tokens": tokens, "offsets": offsets, "labels": labels, "alignable": True})
    except ValueError:
        # Answer text doesn't align to token boundaries we flag it.
        return pd.Series({"tokens": None, "offsets": None, "labels": None, "alignable": False})

#Appending BIO columns to df
bio_results = df.apply(try_bio, axis=1)
df = pd.concat([df, bio_results], axis=1)

print(f"{(~df['alignable']).sum()} / {len(df)} rows fail alignment")
print(df.groupby("answerable")["alignable"].mean())


# Derive the clean training view by filtering out the failing rows
BIO_df = df[df["alignable"]].reset_index(drop=True)
dropped_df = df[~df["alignable"]]  # kept around for reporting


# Round-trip check: after BIO labeling, is the char length the same for tokens and labels?
mismatches = 0
for row in BIO_df.itertuples():
    if row.answerable:
        rt_start, rt_text = bio_to_character_span(row.context, row.offsets, row.labels)
        if rt_start != row.answer_start or rt_text != row.answer:
            mismatches += 1
    else:
        rt_start, rt_text = bio_to_character_span(row.context, row.offsets, row.labels)
        if rt_start is not None or rt_text != "":
            mismatches += 1

print(f"{mismatches} / {len(BIO_df)} rows failed round-trip check")

BIO_df[["context", "answerable", "tokens", "labels"]]

84 / 7490 rows fail alignment
answerable
False    1.000000
True     0.987936
Name: alignable, dtype: float64
0 / 7406 rows failed round-trip check


,context,answerable,tokens,labels
0,The conflict between France and Spain continue...,True,"[The, conflict, between, France, and, Spain, c...","[O, O, O, B-ANS, O, O, O, O, O, O, O, O, O, O,..."
1,"X-rays make up X-radiation, a form of electrom...",True,"[X, -, rays, make, up, X, -, radiation, ,, a, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
2,"In 2022, Beijing will become the first-ever ci...",True,"[In, 2022, ,, Beijing, will, become, the, firs...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
3,The British Broadcasting Corporation (BBC) is ...,True,"[The, British, Broadcasting, Corporation, (, B...","[O, B-ANS, I-ANS, I-ANS, I-ANS, I-ANS, I-ANS, ..."
4,"Palestine ( '), officially the State of Palest...",True,"[Palestine, (, ', ), ,, officially, the, State...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
...,...,...,...,...
7401,Reyyalagadda is a village belonging to Gangara...,True,"[Reyyalagadda, is, a, village, belonging, to, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
7402,Boothumillipadu is a village in Gannavaram man...,True,"[Boothumillipadu, is, a, village, in, Gannavar...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
7403,Mallavemula is a village belonging to Chagalam...,False,"[Mallavemula, is, a, village, belonging, to, C...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
7404,"Andria Urushadze (; born April 25, 1968) is a ...",False,"[Andria, Urushadze, (, ;, born, April, 25, ,, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."


In [ ]:
#A view of the rows where answerable = true but the length of the answer does not match
#with answer_start + len(answer)
dropped_df[["context", "answerable", "answer"]]

,context,answerable,answer
4861,The Australian radio-astronomer Dr John O'Sull...,True,Dr John O'Sullivan with his colleagues Terence...
4873,The Eye of Horus is an ancient Egyptian symbol...,True,Egypt
4899,British society has experienced significant ch...,True,social clas
4913,"The Battle of Leyte Gulf (Filipino: ""Labanan s...",True,The Battle of Leyte Gul
4920,Utilitarianism is an ethical and philosophical...,True,eremy Bentham
...,...,...,...
15474,"Internationally, IIT Delhi was ranked 172 in t...",True,IT Delhi
15701,These letters are believed to have derived fro...,True,Egypt
15901,"By early 2004, the show had settled into a reg...",True,"Keith Boak, Euros Lyn, Joe Ahearne, Brian Gran..."
16116,Early cultures identified celestial oFAncient ...,True,Ancient structures with possibly astronomical ...


In [ ]:
def run_bio_checks():
    checks_passed = 0
    checks_total = 0

    def check(name, condition):
        nonlocal checks_passed, checks_total
        checks_total += 1
        status = "PASS" if condition else "FAIL"
        if condition:
            checks_passed += 1
        print(f"[{status}] {name}")

    #Answer starting at character 0
    context = "Copenhagen is the capital of Denmark."
    answer_text = "Copenhagen"
    answer_start = 0
    tokens, offsets, labels = character_span_to_bio(context, answer_start, answer_text)
    check("answer at char 0: first token is B-ANS", labels[0] == "B-ANS")
    check("answer at char 0: no other tokens are ANS", labels[1:] == ["O"] * (len(labels) - 1))
    rt_start, rt_text = bio_to_character_span(context, offsets, labels)
    check("answer at char 0: round-trips correctly", (rt_start, rt_text) == (answer_start, answer_text))

    #Multi-token answer
    context = "The main character of Jujutsu Kaisen is Itadori Yuji."
    answer_text = "Itadori Yuji"
    answer_start = context.index(answer_text)
    tokens, offsets, labels = character_span_to_bio(context, answer_start, answer_text)
    ans_indices = [i for i, l in enumerate(labels) if l != "O"]
    check("multi-token answer: spans exactly 2 tokens", len(ans_indices) == 2)
    check("multi-token answer: starts with B-ANS", labels[ans_indices[0]] == "B-ANS")
    check("multi-token answer: continues with I-ANS", all(labels[i] == "I-ANS" for i in ans_indices[1:]))
    rt_start, rt_text = bio_to_character_span(context, offsets, labels)
    check("multi-token answer: round-trips correctly", (rt_start, rt_text) == (answer_start, answer_text))

    #Punctuation immediately adjacent to the answer (no whitespace separator)
    context = "She was born in Odense, Denmark."
    answer_text = "Odense"
    answer_start = context.index(answer_text)  # immediately followed by ','
    tokens, offsets, labels = character_span_to_bio(context, answer_start, answer_text)
    ans_indices = [i for i, l in enumerate(labels) if l != "O"]
    check("punctuation-adjacent answer: exactly one ANS token (comma excluded)", len(ans_indices) == 1)
    check("punctuation-adjacent answer: comma token is 'O'", tokens[ans_indices[-1] + 1] == "," and labels[ans_indices[-1] + 1] == "O")
    rt_start, rt_text = bio_to_character_span(context, offsets, labels)
    check("punctuation-adjacent answer: round-trips correctly", (rt_start, rt_text) == (answer_start, answer_text))

    #Unanswerable example (no answer_start / answer_text)
    context = "This string contains no answer for the question."
    tokens, offsets, labels = character_span_to_bio(context, answer_start=None, answer_text="")
    check("unanswerable: all labels are 'O'", all(l == "O" for l in labels))
    check("unanswerable: tokens/offsets still populated", len(tokens) > 0 and len(offsets) == len(tokens))
    rt_start, rt_text = bio_to_character_span(context, offsets, labels)
    check("unanswerable: round-trips to (None, '')", (rt_start, rt_text) == (None, ""))

    print(f"\n{checks_passed}/{checks_total} checks passed")
    assert checks_passed == checks_total, "One or more BIO conversion checks failed"

run_bio_checks()

[PASS] answer at char 0: first token is B-ANS
[PASS] answer at char 0: no other tokens are ANS
[PASS] answer at char 0: round-trips correctly
[PASS] multi-token answer: spans exactly 2 tokens
[PASS] multi-token answer: starts with B-ANS
[PASS] multi-token answer: continues with I-ANS
[PASS] multi-token answer: round-trips correctly
[PASS] punctuation-adjacent answer: exactly one ANS token (comma excluded)
[PASS] punctuation-adjacent answer: comma token is 'O'
[PASS] punctuation-adjacent answer: round-trips correctly
[PASS] unanswerable: all labels are 'O'
[PASS] unanswerable: tokens/offsets still populated
[PASS] unanswerable: round-trips to (None, '')

13/13 checks passed


In [ ]:
BIO_df["question_en"] = BIO_df.apply(lambda row: translate(row["question"], row["lang"]),axis=1)

In [ ]:
#CODE CELL WAS IMPORTED FROM lab2.ipynb
def token_features(tokens, index):
    word = tokens[index]

    features = {
        "bias": 1.0,
        "word.lower": word.lower(),
        "word.prefix2": word[:2].lower(),
        "word.suffix2": word[-2:].lower(),
        "word.suffix3": word[-3:].lower(),
        "word.istitle": word.istitle(),
        "word.isupper": word.isupper(),
        "word.isdigit": word.isdigit(),
        "contains_hyphen": "-" in word,
    }

    if index == 0:
        features["BOS"] = True
    else:
        previous = tokens[index - 1]
        features.update({
            "previous.lower": previous.lower(),
            "previous.istitle": previous.istitle(),
            "previous.isupper": previous.isupper(),
        })

    if index == len(tokens) - 1:
        features["EOS"] = True
    else:
        following = tokens[index + 1]
        features.update({
            "next.lower": following.lower(),
            "next.istitle": following.istitle(),
            "next.isupper": following.isupper(),
        })

    return features

In [ ]:
MONTH_NAMES = {"january", "february", "march", "april", "may", "june", "july",
    "august", "september", "october", "november", "december",}

def token_type_flags(token):
    return {
        "is_digit": token.isdigit(),
        "is_year_like": bool(re.fullmatch(r"(1|2)\d{3}", token)),
        "is_month_name": token.lower() in MONTH_NAMES,
        "is_capitalized_word": (token[:1].isupper() and token[1:].islower() if len(token) > 1 else token.isupper()),}

def qtype_match_feature(token, qtype):
    flags = token_type_flags(token)
    if qtype == "number":
        return flags["is_digit"]
    if qtype == "date":
        return flags["is_year_like"] or flags["is_month_name"] or flags["is_digit"]
    if qtype in ("person", "location"):
        return flags["is_capitalized_word"]
    return False


def token_features_qc(tokens, index, overlap_flags, qtype="other"):
    features = token_features(tokens, index)
    features["in_question"] = overlap_flags[index]
    features["prev_in_question"] = overlap_flags[index - 1] if index > 0 else False
    features["next_in_question"] = overlap_flags[index + 1] if index < len(tokens) - 1 else False

    window = 3
    lo, hi = max(0, index - window), min(len(tokens), index + window + 1)
    features["window_overlap_count"] = sum(overlap_flags[lo:hi])
    features["question_type"] = qtype
    features["qtype_match"] = qtype_match_feature(tokens[index], qtype)

    return features

In [ ]:
from functools import lru_cache

@lru_cache(maxsize=200000)
def word_subword_pieces(word):
    pieces = tokenizer.tokenize(word.lower())
    return frozenset(p for p in pieces if p not in string.punctuation)

def compute_overlap_flags(tokens, q_subwords):
    if not q_subwords:
        return [False] * len(tokens)
    return [bool(word_subword_pieces(tok) & q_subwords) for tok in tokens]


In [ ]:
def featurize_bio_df(df):
    features = []
    labels = []
    lengths = []

    for row in df.to_dict("records"):
        tokens = row["tokens"]
        question = row.get("question_en", row.get("question"))
        q_subwords = clean_tokens(question)
        overlap_flags = compute_overlap_flags(tokens, q_subwords)
        qtype = detect_question_type(question) #reuse question-type detector

        lengths.append(len(tokens))
        for i in range(len(tokens)):
            features.append(token_features_qc(tokens, i, overlap_flags, qtype))
        labels.extend(row["labels"])

    return features, labels, lengths


In [ ]:
train_df = BIO_df[BIO_df["split"] == "train"].copy()
val_df = BIO_df[BIO_df["split"] == "validation"].copy()

In [ ]:
train_features, train_labels, train_lengths = featurize_bio_df(train_df)
val_features, val_labels, val_lengths = featurize_bio_df(val_df)

In [ ]:
from sklearn.feature_extraction import DictVectorizer

vectorizer = DictVectorizer(sparse=True)
X_train = vectorizer.fit_transform(train_features)
X_val = vectorizer.transform(val_features)

In [ ]:
from sklearn.linear_model import SGDClassifier

qc_classifier = SGDClassifier(
    loss="log_loss",
    alpha=1e-5,
    class_weight="balanced",
    max_iter=30,
    random_state=42
)

qc_classifier.fit(X_train, train_labels)

SGDClassifier(alpha=1e-05, class_weight='balanced', loss='log_loss',
              max_iter=30, random_state=42)

In [ ]:
def decode_best_span_greedy(predicted_labels):
    start_index = next((i for i, label in enumerate(predicted_labels) if label == "B-ANS"), None)
    if start_index is None:
        return None
    end_index = start_index
    while end_index + 1 < len(predicted_labels) and predicted_labels[end_index + 1] == "I-ANS":
        end_index += 1
    return start_index, end_index + 1


def decode_best_span_scored(probs, classes, max_span_len=25, eps=1e-12):
    b_idx = classes.index("B-ANS")
    i_idx = classes.index("I-ANS")
    o_idx = classes.index("O")

    n = len(probs)
    log_o = np.log(probs[:, o_idx] + eps)
    log_b = np.log(probs[:, b_idx] + eps)
    log_i = np.log(probs[:, i_idx] + eps)

    best_delta = 0.0
    best_span = None

    for start in range(n):
        running_delta = log_b[start] - log_o[start]
        if running_delta > best_delta:
            best_delta = running_delta
            best_span = (start, start + 1)
        end_limit = min(n, start + max_span_len)
        for end in range(start + 1, end_limit):
            running_delta += log_i[end] - log_o[end]
            if running_delta > best_delta:
                best_delta = running_delta
                best_span = (start, end + 1)

    return best_span

def span_to_bio_labels(n_tokens, span):
    labels = ["O"] * n_tokens
    if span is None:
        return labels
    start, end = span
    labels[start] = "B-ANS"
    for i in range(start + 1, end):
        labels[i] = "I-ANS"
    return labels


In [ ]:
def predict_span_qc_classifier(row, vectorizer, classifier):
    tokens, offsets = row["tokens"], row["offsets"]
    question = row.get("question_en", row.get("question"))
    overlap_flags = compute_overlap_flags(tokens, clean_tokens(question))
    qtype = detect_question_type(question)  # FIXED

    features = [token_features_qc(tokens, i, overlap_flags, qtype) for i in range(len(tokens))]
    X = vectorizer.transform(features)
    probs = classifier.predict_proba(X)
    span = decode_best_span_scored(probs, list(classifier.classes_))

    if span is None:
        return None, ""
    start_index, end_index = span
    char_start, char_end = offsets[start_index][0], offsets[end_index - 1][1]
    return char_start, row["context"][char_start:char_end]


In [ ]:
def empty_baseline(question, context):
    return None, ""

In [ ]:
def predict_bio_labels(row, vectorizer, classifier):
    tokens = row["tokens"]
    question = row.get("question_en", row.get("question"))
    overlap_flags = compute_overlap_flags(tokens, clean_tokens(question))
    qtype = detect_question_type(question)

    features = [token_features_qc(tokens, i, overlap_flags, qtype) for i in range(len(tokens))]
    X = vectorizer.transform(features)

    probs = classifier.predict_proba(X)
    span = decode_best_span_scored(probs, list(classifier.classes_))

    return span_to_bio_labels(len(tokens), span)


In [ ]:
def bio_spans(labels):
    spans = set()
    start = None
    for i, label in enumerate(labels + ["O"]):
        if label == "B-ANS":
            if start is not None:
                spans.add((start, i))
            start = i

        elif label == "I-ANS":
            if start is None:
                start = i

        else:
            if start is not None:
                spans.add((start, i))
                start = None

    return spans


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

def token_level_scores(gold_sequences, predicted_sequences):
    gold_tokens, pred_tokens = [], []
    for gold, predicted in zip(gold_sequences, predicted_sequences):
        gold_tokens.extend([1 if l in {"B-ANS", "I-ANS"} else 0 for l in gold])
        pred_tokens.extend([1 if l in {"B-ANS", "I-ANS"} else 0 for l in predicted])
    return {
        "precision": precision_score(gold_tokens, pred_tokens, zero_division=0),
        "recall": recall_score(gold_tokens, pred_tokens, zero_division=0),
        "f1": f1_score(gold_tokens, pred_tokens, zero_division=0),
    }

def exact_span_match_rate(gold_sequences, predicted_sequences):
    matches = [bio_spans(g) == bio_spans(p) for g, p in zip(gold_sequences, predicted_sequences)]
    return sum(matches) / len(matches)

def answerability_agreement(gold_sequences, predicted_sequences):
    # Did the model correctly decide an answer exists (or doesn't), regardless of exact boundaries?
    correct = [bool(bio_spans(g)) == bool(bio_spans(p)) for g, p in zip(gold_sequences, predicted_sequences)]
    return sum(correct) / len(correct)


In [ ]:
gold_bio_sequences = [row["labels"] for row in val_df.to_dict("records")]
predicted_bio_sequences = [predict_bio_labels(row, vectorizer, qc_classifier) for row in val_df.to_dict("records")]
empty_bio_sequences = [["O"] * len(g) for g in gold_bio_sequences]

qc_scores = token_level_scores(gold_bio_sequences, predicted_bio_sequences)
qc_exact = exact_span_match_rate(gold_bio_sequences, predicted_bio_sequences)
qc_answerability = answerability_agreement(gold_bio_sequences, predicted_bio_sequences)

empty_scores = token_level_scores(gold_bio_sequences, empty_bio_sequences)
empty_exact = exact_span_match_rate(gold_bio_sequences, empty_bio_sequences)
empty_answerability = answerability_agreement(gold_bio_sequences, empty_bio_sequences)

print("QC model - precision: {:.4f}  recall: {:.4f}  F1: {:.4f}  exact span: {:.4f}  answerability agreement: {:.4f}".format(
    qc_scores["precision"], qc_scores["recall"], qc_scores["f1"], qc_exact, qc_answerability))
print("Empty baseline - precision: {:.4f}  recall: {:.4f}  F1: {:.4f}  exact span: {:.4f}  answerability agreement: {:.4f}".format(
    empty_scores["precision"], empty_scores["recall"], empty_scores["f1"], empty_exact, empty_answerability))


QC model - precision: 0.3193  recall: 0.1349  F1: 0.1897  exact span: 0.1922  answerability agreement: 0.7783
Empty baseline - precision: 0.0000  recall: 0.0000  F1: 0.0000  exact span: 0.1426  answerability agreement: 0.1426


# Week 38:


In [ ]:
!pip install collections

ERROR: Could not find a version that satisfies the requirement collections (from versions: none)
ERROR: No matching distribution found for collections


In [ ]:
from collections import Counter

ko_df = df[df["lang"] == "ko"]
df_groups = ko_df.groupby("split")

PAD, UNK, BOS, EOS = "<pad>", "<unk>", "<bos>", "<eos>"

def subword_pieces(text):
  return tokenizer.tokenize(str(text).lower())

train_piece_counts = Counter()
for q in df_groups.get_group("train")["question"]:
  train_piece_counts.update(subword_pieces(q))

#Get the vocabulary from the training questions and string to index
# + index to string, so we can provide n word context to model and baseline
vocab = [PAD, UNK, BOS, EOS] + sorted(train_piece_counts)
stoi = {tok: i for i, tok in enumerate(vocab)}
itos = {i: tok for tok, i in stoi.items()}

def encode(text):
  ids = [stoi.get(p, stoi[UNK]) for p in subword_pieces(text)]
  return [stoi[BOS]] + ids + [stoi[EOS]]

val_quest = df_groups.get_group("validation")["question"]
n_val_oov = sum(1 for q in val_quest for p in subword_pieces(q) if p not in stoi)
n_val_pieces = sum(len(subword_pieces(q)) for q in val_quest)
print(f"Vocav size in Korean training: {len(vocab)}")
print(f"Validation OOV rate: {n_val_oov}/{n_val_pieces} pieces "
      f"({100*n_val_oov/n_val_pieces:.2f}%) mapped to <unk>")

Vocav size in Korean training: 2227
Validation OOV rate: 114/4329 pieces (2.63%) mapped to <unk>


Unigram Baseline

In [24]:
import math

def train_unigram():
    counts = Counter()
    for q in df_groups.get_group("train")["question"]:
        counts.update(encode(q))
    total = sum(counts.values())
    V = len(vocab)
    return {i: (counts.get(i, 0) + 1) / (total + V) for i in range(V)}

unigram_probs = train_unigram()

def unigram_perplexity(text, probs):
    ids = encode(text)[1:]
    nll = -sum(math.log(probs[i]) for i in ids)
    return math.exp(nll / len(ids)), len(ids)

### Andreas Toftager Ejlersen (knm882) RNN Implementation:


In [ ]:
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence

device = "cuda" if torch.cuda.is_available() else "cpu"

class RNNLM(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, hidden_dim=256):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=stoi[PAD])
        self.rnn = nn.RNN(emb_dim, hidden_dim, batch_first=True)
        self.out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        h, _ = self.rnn(self.emb(x))
        return self.out(h)

RNNmodel = RNNLM(len(vocab)).to(device)

## Shared Tests and training


In [ ]:
def make_batch(texts, batch_size=32):
    ids = [torch.tensor(encode(t)) for t in texts]
    for i in range(0, len(ids), batch_size):
        yield pad_sequence(ids[i:i+batch_size], batch_first=True, padding_value=stoi[PAD])

#Training neural network via gradient descent
optimizer = torch.optim.Adam(RNNmodel.parameters(), lr=1e-3)

#we ignore padding tokens to not penalize the model for meaningless predictions.
criterion = nn.CrossEntropyLoss(ignore_index=stoi[PAD])
train_texts = df_groups.get_group("train")["question"].tolist()

#HEBA YOU CAN USE MY TRAINING BATCH IF YOU NEED TO
#THIS IS PROBABLY ONLY USEFUL IF YOU CHOOSE A NEURAL NETWORK

#running training 5 times with same questions.
RNNmodel.train()
for epoch in range(5):
    total_loss, n_batches = 0.0, 0
    for batch in make_batch(train_texts):
        batch = batch.to(device)
        #predict each token with the context of everything before.
        inputs, targets = batch[:, :-1], batch[:, 1:]
        logits = RNNmodel(inputs)
        loss = criterion(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1
    print(f"epoch {epoch+1}: avg loss {total_loss/n_batches:.4f}")

epoch 1: avg loss 5.1758
epoch 2: avg loss 3.9997
epoch 3: avg loss 3.5451
epoch 4: avg loss 3.2131
epoch 5: avg loss 2.9421


In [ ]:
@torch.no_grad()
def rnn_perplexity(text, model):
    model.eval()
    ids = torch.tensor(encode(text)).unsqueeze(0).to(device)
    inputs, targets = ids[:, :-1], ids[:, 1:]
    logits = model(inputs)
    loss = nn.functional.cross_entropy(
        logits.reshape(-1, logits.size(-1)), targets.reshape(-1), reduction="sum"
    )
    return math.exp(loss.item() / targets.numel()), targets.numel()

val_texts = df_groups.get_group("validation")["question"].tolist()
#HEBA PUT YOUR MODEL HERE!
rnn_ppls = [rnn_perplexity(t, model)[0] for t in val_texts]
uni_ppls = [unigram_perplexity(t, unigram_probs)[0] for t in val_texts]

print(f"RNN mean validation perplexity:     {sum(rnn_ppls)/len(rnn_ppls):.2f}")
print(f"Unigram mean validation perplexity: {sum(uni_ppls)/len(uni_ppls):.2f}")

results = sorted(zip(val_texts, rnn_ppls), key=lambda x: x[1])

print("RNN Lowest perplexity:")
for text, ppl in results[:5]:
    print(f"  {ppl:.1f}  {text}  ->  {translate_cached(text, 'ko')}")

print("\nRNN Highest perplexity:")
for text, ppl in results[-5:]:
    print(f"  {ppl:.1f}  {text}  ->  {translate_cached(text, 'ko')}")

RNN mean validation perplexity:     72.52
Unigram mean validation perplexity: 371.52
RNN Lowest perplexity:
  2.3  중국에서 가장 인구밀도가 높은 도시는 어디인가?  ->  What is the most densely populated city in China?
  3.0  중국에서 가장 작은 도시는 어디인가?  ->  Where is the smallest city in China?
  3.2  프랑스의 제 1대 대통령은 누구인가요?  ->  Who was the first president of France?
  3.2  프랑스의 제 1대 대통령은 누구인가요?  ->  Who was the first president of France?
  3.3  세상에서 가장 큰 대학교는 무엇인가?  ->  What is the largest university in the world?

RNN Highest perplexity:
  455.0  논증의 오류로 인해 무지에의 호소 오류를 범할수 있는가?  ->  Is it possible to make an unknown appeal because of an error in the argument?
  548.0  임시정부는 종전의 정부가 무너진 후, 무정부 상태를 해소하기 위해 임시로 구성된 정부를 말하나요?  ->  Does the interim government say that it is a temporary government to dismantle the state of non-government after the collapse of the former government?
  730.3  탄광 노동자들이 파업을 한 이유는 무엇인가?  ->  What was the reason for the strike?
  1254.3  니콜라 테슬라와 Tesla, Inc.은 관계가 있는가?  ->  Is there a connect

In [ ]:
text = "니콜라 테슬라와 Tesla, Inc.은 관계가 있는가?"
pieces = subword_pieces(text)
for p in pieces:
    print(f"{p!r:20s} in vocab: {p in stoi}")

'▁니'                 in vocab: True
'콜라'                 in vocab: False
'▁테'                 in vocab: True
'슬'                  in vocab: True
'라'                  in vocab: True
'와'                  in vocab: True
'▁tes'               in vocab: False
'la'                 in vocab: False
','                  in vocab: True
'▁in'                in vocab: False
'c'                  in vocab: True
'.'                  in vocab: True
'은'                  in vocab: True
'▁관계'                in vocab: True
'가'                  in vocab: True
'▁있는'                in vocab: True
'가'                  in vocab: True
'?'                  in vocab: True


latin language tokens give 100% failure rate since they are not in the training set. The hypothesis that foreign language will tend to plummet performance seems to be true.

What is the "SOMETHING" in "COUNTY" is a question with low perplexity. Hypothesis is that due to short sentence length and frequent sighting of this sentence structure in training that the RNN predicts this type of sentence well.


Smoothed n-gram class

In [42]:
import math
from collections import defaultdict, Counter

class AddAlphaNGram:
    def __init__(self, n, alpha):
        self.n, self.alpha, self.V = n, alpha, len(vocab)
        self.c = defaultdict(Counter)
        self.h = Counter()

    def _seq(self, text):
        return [stoi[BOS]] * max(self.n - 2, 0) + encode(text)

    def _positions(self, ids):
        return range(max(self.n - 1, 1), len(ids))

    def fit(self, texts):
        for t in texts:
            ids = self._seq(t)
            for i in self._positions(ids):
                hist = tuple(ids[i - (self.n - 1): i])
                self.c[hist][ids[i]] += 1
                self.h[hist] += 1
        return self

    def prob(self, hist, w):
        return (self.c[hist][w] + self.alpha) / (self.h[hist] + self.alpha * self.V)

    def nll(self, text):
        ids = self._seq(text)
        total, count = 0.0, 0
        for i in self._positions(ids):
            hist = tuple(ids[i - (self.n - 1): i])
            total -= math.log(self.prob(hist, ids[i]))
            count += 1
        return total, count

In [43]:
def corpus_ppl(model, texts):
    nll, n = 0.0, 0
    for t in texts:
        a, b = model.nll(t)
        nll += a
        n += b
    return math.exp(nll / n)

def per_question_ppl(model, texts):
    out = []
    for t in texts:
        a, b = model.nll(t)
        out.append(math.exp(a / b))
    return out

In [47]:
#simpel grid search
results={}
for n in (2,3,4):
    for alpha in (1.0, 0.1, 0.01, 0.001):
        m = AddAlphaNGram(n, alpha).fit(train_texts)
        results[(n, alpha)] = corpus_ppl(m, val_texts)
        print(f"n={n} alpha={alpha:<6} val perplexity = {results[(n, alpha)]:.1f}")

best_key = min(results, key=results.get)
print("\nbest(n, alpha):", best_key, round(results[best_key], 1))

n=2 alpha=1.0    val perplexity = 186.7
n=2 alpha=0.1    val perplexity = 69.1
n=2 alpha=0.01   val perplexity = 43.8
n=2 alpha=0.001  val perplexity = 48.8
n=3 alpha=1.0    val perplexity = 413.6
n=3 alpha=0.1    val perplexity = 153.2
n=3 alpha=0.01   val perplexity = 74.9
n=3 alpha=0.001  val perplexity = 57.5
n=4 alpha=1.0    val perplexity = 700.6
n=4 alpha=0.1    val perplexity = 301.0
n=4 alpha=0.01   val perplexity = 149.4
n=4 alpha=0.001  val perplexity = 106.2

best (n, alpha): (2, 0.01) 43.8


In [49]:
best_model=AddAlphaNGram(best_key[0], best_key[1]).fit(train_texts)

In [40]:
def unigram_corpus_ppl(texts):
    log_sum, n_tokens = 0.0, 0
    for t in texts:
        ppl, n = unigram_perplexity(t, unigram_probs)
        log_sum += math.log(ppl) * n
        n_tokens += n
    return math.exp(log_sum / n_tokens)

print(f"Unigram(baseline):{unigram_corpus_ppl(val_texts):.1f}")
print(f"Bigram (add-{best_key[1]}):{corpus_ppl(best_model, val_texts):.1f}")

Unigram(baseline):344.8
Bigram (add-0.01):43.8


In [50]:
bi_ppls= per_question_ppl(best_model, val_texts)
ranked=sorted(zip(val_texts,bi_ppls), key=lambda x: x[1])

print("BIGRAM lowest perplexity:")
for text, ppl in ranked[:5]:
    print(f"{ppl:.1f}{text}->{translate_cached(text, 'ko')}")

print("\BIGRAM highest perplexity:")
for text, ppl in ranked[-5:]:
    print(f"{ppl:.1f}{text}->{translate_cached(text, 'ko')}")

BIGRAM lowest perplexity:
4.1중국에서 가장 인구밀도가 높은 도시는 어디인가?->What is the most densely populated city in China?
4.2세상에서 가장 큰 대학교는 무엇인가?->What is the largest university in the world?
4.4세상에서 가장 오래된 대학교는 무엇인가?->What is the oldest university in the world?
4.4세상에서 가장 오래된 대학교는 무엇인가?->What is the oldest university in the world?
4.4세상에서 가장 오래된 대학교는 무엇인가?->What is the oldest university in the world?
\BIGRAM highest perplexity:
556.8인체가 자외선에 많이 노출된다면 어떤 변화가 일어나는가?->What changes do you make when you're exposed to UV rays?
604.4냉전 기간 동안 미국을 돕는 나라는?->The country that helped America during the Cold War?
609.8중동에서 사람들은 어떤 언어를 말하는가?->What language do people speak in the Middle East?
659.7제1대 휘트워스 준남작 조지프 휘트워스 경의 소속은 어디인가요?->Where is the first Whitworth deputy earl, Sir George Whitworth?
724.8임시정부는 종전의 정부가 무너진 후, 무정부 상태를 해소하기 위해 임시로 구성된 정부를 말하나요?->Does the interim government say that it is a temporary government to dismantle the state of non-government after the collapse of the former government?


In [38]:
def token_surprisals(model, text):
    ids = model._seq(text)
    rows = []
    for i in model._positions(ids):
        hist = tuple(ids[i - (model.n - 1): i])
        p = model.prob(hist, ids[i])
        rows.append((itos[ids[i]], -math.log2(p), model.h[hist]))
    return rows

example = ranked[-1][0]
print(example)
for tok, bits, ctx_count in token_surprisals(best_model, example):
    print(f"{tok!r:15s} surprisal={bits:5.1f} bits   context seen {ctx_count}x in training")

임시정부는 종전의 정부가 무너진 후, 무정부 상태를 해소하기 위해 임시로 구성된 정부를 말하나요?
'▁임'            surprisal=  9.3 bits   context seen 2422x in training
'시'             surprisal=  4.9 bits   context seen 8x in training
'정부'            surprisal=  6.0 bits   context seen 43x in training
'는'             surprisal= 11.3 bits   context seen 3x in training
'▁종'            surprisal= 16.8 bits   context seen 1154x in training
'전'             surprisal= 11.9 bits   context seen 17x in training
'의'             surprisal=  6.2 bits   context seen 50x in training
'<unk>'         surprisal= 15.9 bits   context seen 601x in training
'▁무'            surprisal= 11.1 bits   context seen 0x in training
'너'             surprisal=  2.8 bits   context seen 27x in training
'진'             surprisal= 11.8 bits   context seen 13x in training
'▁후'            surprisal= 12.3 bits   context seen 29x in training
','             surprisal= 12.0 bits   context seen 19x in training
'▁무'            surprisal= 12.1 bits   context seen 23x in 

Hypothesis: An n-gram language model assigns lower perplexity to text containing familiar local contexts that were frequently observed during training, while unfamiliar or unseen contexts lead to higher surprisal and therefore higher perplexity.